In [ ]:
import pandas as pd
df=pd.read_csv('/Combined .csv')

FileNotFoundError: [Errno 2] No such file or directory: '/Combined .csv'

In [ ]:
df.head()

,Sr. No.,News Items,Label
0,1,ٹی ٹی پی نے پنجاب حکومت کے ہیلی کاپٹر کے عملے ...,FAKE
1,2,مارک زکربرگ سیاست میں آنے کا سوچ رہے ہیں۔,FAKE
2,3,فریدہ جلال نے اپنی موت کی افواہوں پر تنقید کی۔,FAKE
3,4,جعلی خبریں: پاپ اسٹار حدیقہ کیانی نے جعلی منشی...,FAKE
4,5,صنم ماروی نے میڈیا پر گردش کرنے والی زیادتی او...,FAKE


In [ ]:
# Install required packages (only if not already installed)
!pip install deep-translator transformers torch

# Import necessary libraries
from deep_translator import GoogleTranslator
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd  # Easier data handling

# Load the pre-trained sentiment model
tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

# Use GPU if available, else fallback to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)



# Extract the Urdu news text column
texts = df['News Items'].tolist()

# Empty list to store sentiment labels
sentiments = []

# Loop through each text item to analyze sentiment
for index, text in enumerate(texts):
    try:
        # Step 1: Translate Urdu to English
        translated_text = GoogleTranslator(source="ur", target="en").translate(text)

        # Step 2: Tokenize the translated text
        inputs = tokenizer(translated_text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        inputs = {key: tensor.to(device) for key, tensor in inputs.items()}

        # Step 3: Get prediction from model
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=1)
        prediction = torch.argmax(probabilities, dim=1).item()

        # Step 4: Convert original 5-class output to 3-class
        if prediction in [0, 1]:          # Very Negative or Negative
            sentiments.append("Negative")
        elif prediction == 2:             # Neutral
            sentiments.append("Neutral")
        else:                             # Positive or Very Positive
            sentiments.append("Positive")

    except Exception as e:
        sentiments.append("Error")  # Mark error entries
        print(f"Error at index {index}: {e}")

# Add sentiment labels to original DataFrame
df["Sentiment"] = sentiments

# Save the new labeled dataset to CSV
df.to_csv("urdu_news_with_sentiment.csv", index=False)
print("Dataset saved as 'urdu_news_with_sentiment.csv'")


Error at index 1044: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 1714: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 1940: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 2729: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 2731: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 2736: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 2739: Request exception can happen due to an api connection error. Please check your connection and try again
Error at index 2746: Request exception can happen due to an api connection error. Please check your connection and try again


In [ ]:
from google.colab import files
files.download("urdu_news_with_sentiment.csv")


FileNotFoundError: Cannot find file: urdu_news_with_sentiment.csv

In [ ]:
import pandas as pd
df2=pd.read_csv('/content/urdu_news_with_sentiment.csv',encoding='utf-8-sig')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x92 in position 565: invalid start byte

In [ ]:
df2.head()

,Sr. No.,News Items,Label,Sentiment
0,1.0,?? ?? ?? ?? ????? ????? ?? ???? ????? ?? ???? ...,FAKE,Negative
1,2.0,???? ?????? ????? ??? ??? ?? ??? ??? ????,FAKE,Positive
2,3.0,????? ???? ?? ???? ??? ?? ??????? ?? ????? ???,FAKE,Negative
3,4.0,???? ?????: ??? ????? ????? ????? ?? ???? ????...,FAKE,Negative
4,5.0,??? ????? ?? ????? ?? ???? ???? ???? ?????? ??...,FAKE,Negative


In [ ]:
df2['Sentiment'].unique()

array(['Negative', 'Positive', 'Neutral', 'Error'], dtype=object)

In [ ]:
df2['Sentiment'].describe()

,Sentiment
count,10084
unique,4
top,Negative
freq,6431


In [ ]:
df2['Sentiment'].value_counts()

,count
Sentiment,
Negative,6431
Positive,2728
Neutral,797
Error,128
